In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from keras.layers import TextVectorization
from tensorflow import Tensor


# =============================================================================
# Paths
# =============================================================================

CAPTIONS_DIR = Path("../data/captions")
FLICKR8K_TEXT_DIR = Path("../Flickr8k/Flickr8k_text")

CAPTIONS_FILE = CAPTIONS_DIR / "captions.csv"


# =============================================================================
# Split captions
# =============================================================================

def create_caption_subset(
    captions_file: str | Path,
    images_file: str | Path,
    output_file: str | Path,
) -> None:

    captions_df = pd.read_csv(captions_file)

    images_df = pd.read_csv(
        images_file,
        header=None,
        names=["image"],
    )

    images_df["image"] = images_df["image"].str.replace(
        ".jpg",
        "",
        regex=False,
    )

    (
        images_df
        .merge(captions_df, on="image", how="inner")
        .to_csv(output_file, index=False)
    )


def split_caption_file(
    captions_file: str | Path,
    splits: list[tuple[str | Path, str | Path]],
) -> None:

    for images_file, output_file in splits:
        create_caption_subset(
            captions_file,
            images_file,
            output_file,
        )


# =============================================================================
# Caption loading
# =============================================================================

def load_captions(path: str | Path) -> dict[str, list[str]]:

    df = pd.read_csv(path)

    return (
        df.groupby("image")["caption"]
        .apply(list)
        .to_dict()
    )


# =============================================================================
# Caption preprocessing
# =============================================================================

def compute_max_len(
    captions: dict[str, list[str]],
    percentile: int =97,
) -> int:

    lengths = [
        len(caption.split())
        for image_captions in captions.values()
        for caption in image_captions
    ]

    return int(np.percentile(lengths, percentile))


def truncate_captions(
    captions: dict[str, list[str]],
    max_len: int,
) -> tuple[dict[str, list[str]], int]:

    truncated = {}
    truncated_count = 0

    for image, image_captions in captions.items():

        processed = []

        for caption in image_captions:

            words = caption.split()

            if len(words) > max_len:
                processed.append(" ".join(words[:max_len]))
                truncated_count += 1
            else:
                processed.append(caption)

        truncated[image] = processed

    return truncated, truncated_count


def add_start_end_tokens(
    captions: dict[str, list[str]],
) -> dict[str, list[str]]:

    return {
        image: [
            f"[START] {caption} [END]"
            for caption in image_captions
        ]
        for image, image_captions in captions.items()
    }


# =============================================================================
# Vectorization
# =============================================================================

def create_vectorizer(
    captions: dict[str, list[str]],
    output_sequence_length: int,
) -> TextVectorization:

    vectorizer = TextVectorization(
        output_mode="int",
        split="whitespace",
        standardize=None,      # type: ignore
        output_sequence_length=output_sequence_length,
        name="text_vectorization",
    )

    caption_list = [
        caption
        for image_captions in captions.values()
        for caption in image_captions
    ]

    vectorizer.adapt(caption_list)

    return vectorizer


def vectorize_captions(
    vectorizer: TextVectorization,
    captions: dict[str, list[str]],
) -> Tensor:

    caption_list = [
        caption
        for image_captions in captions.values()
        for caption in image_captions
    ]

    return vectorizer(caption_list)


def split_teacher_forcing(
    captions: Tensor,
) -> tuple[Tensor, Tensor]:

    return captions[:, :-1], captions[:, 1:] # type: ignore


# =============================================================================
# Dataset
# =============================================================================

def build_caption_dataset(
    captions: dict[str, list[str]],
    input_tensor: Tensor,
    target_tensor: Tensor,
):
    dataset = []

    idx = 0

    for image, image_captions in captions.items():

        for _ in image_captions:

            dataset.append({
                "image": image,
                "input": input_tensor[idx], # type: ignore
                "target": target_tensor[idx], # type: ignore
            })

            idx += 1

    return dataset


# =============================================================================
# TFRecord
# =============================================================================

def create_tf_record(
    dataset,
    output_file: str | Path,
) -> None:

    with tf.io.TFRecordWriter(str(output_file)) as writer:

        for sample in dataset:

            image = sample["image"]
            input_seq = sample["input"].numpy().astype(np.int64)
            target_seq = sample["target"].numpy().astype(np.int64)

            example = tf.train.Example(
                features=tf.train.Features(
                    feature={
                        "image": tf.train.Feature(
                            bytes_list=tf.train.BytesList(
                                value=[image.encode()]
                            )
                        ),
                        "input": tf.train.Feature(
                            int64_list=tf.train.Int64List(
                                value=input_seq.tolist()
                            )
                        ),
                        "target": tf.train.Feature(
                            int64_list=tf.train.Int64List(
                                value=target_seq.tolist()
                            )
                        ),
                    }
                )
            )

            writer.write(example.SerializeToString())


def preprocess_split(
    captions_file: str | Path,
    output_file: str | Path,
    max_len: int,
    vectorizer: TextVectorization | None = None,
) -> tuple[
    dict[str, dict[str, Tensor]],
    TextVectorization,
]:

    captions = load_captions(captions_file)

    captions, truncated = truncate_captions(
        captions,
        max_len,
    )

    print(f"{captions_file}: truncated {truncated} captions")

    captions = add_start_end_tokens(captions)

    if vectorizer is None:
        vectorizer = create_vectorizer(
            captions,
            output_sequence_length=max_len + 2,
        )

        print(f"Vocabulary size: {len(vectorizer.get_vocabulary())}")

    captions_tensor = vectorize_captions(
        vectorizer,
        captions,
    )

    captions_input, captions_target = split_teacher_forcing(
        captions_tensor,
    )

    dataset = build_caption_dataset(
        captions,
        captions_input,
        captions_target,
    )

    create_tf_record(
        dataset,
        output_file,
    )

    return dataset, vectorizer

2026-06-28 17:46:36.473776: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
split_caption_file(
    captions_file=CAPTIONS_FILE,
    splits=[
        (
            FLICKR8K_TEXT_DIR / "Flickr_8k.trainImages.txt",
            CAPTIONS_DIR / "train_captions.csv",
        ),
        (
            FLICKR8K_TEXT_DIR / "Flickr_8k.devImages.txt",
            CAPTIONS_DIR / "val_captions.csv",
        ),
        (
            FLICKR8K_TEXT_DIR / "Flickr_8k.testImages.txt",
            CAPTIONS_DIR / "test_captions.csv",
        ),
    ],
)

print("Caption files created.")

Caption files created.


In [3]:
train_captions = load_captions(
    CAPTIONS_DIR / "train_captions.csv"
)

max_len = compute_max_len(
    train_captions,
    percentile=97,
)

print(f"Max length: {max_len}")

train_dataset, vectorizer = preprocess_split(
    captions_file=CAPTIONS_DIR / "train_captions.csv",
    output_file="../data/records/train.tfrecord",
    max_len=max_len,
)

val_dataset, _ = preprocess_split(
    captions_file=CAPTIONS_DIR / "val_captions.csv",
    output_file="../data/records/val.tfrecord",
    max_len=max_len,
    vectorizer=vectorizer,
)

test_dataset, _ = preprocess_split(
    captions_file=CAPTIONS_DIR / "test_captions.csv",
    output_file="../data/records/test.tfrecord",
    max_len=max_len,
    vectorizer=vectorizer,
)

print("TFRecords created successfully.")

Max length: 15
../data/captions/train_captions.csv: truncated 631 captions
Vocabulary size: 7578
../data/captions/val_captions.csv: truncated 117 captions
../data/captions/test_captions.csv: truncated 85 captions
TFRecords created successfully.
